# Kısıtlı Optimizasyon – Lagrange Çarpanları (Newton–KKT Yaklaşımı)

## Problem Tanımı

Amaç:

$$
\min_{x,y} f(x,y)
$$

Kısıt:

$$
g(x,y) = 0
$$

---

## Lagrangian Fonksiyonu

Kısıtlı problemi çözmek için Lagrangian tanımlanır:

$$
\mathcal{L}(x,y,\lambda) = f(x,y) + \lambda g(x,y)
$$

Burada $\lambda$ Lagrange çarpanıdır.

---

## Birinci Mertebe Optimalite Koşulları (KKT)

Optimal noktada şu koşul sağlanır:

$$
\nabla \mathcal{L}(x^*, y^*, \lambda^*) = 0
$$

Bu da aşağıdaki sistemi verir:

$$
\frac{\partial \mathcal{L}}{\partial x} = 0
$$

$$
\frac{\partial \mathcal{L}}{\partial y} = 0
$$

$$
\frac{\partial \mathcal{L}}{\partial \lambda} = g(x,y) = 0
$$

---

## Newton Sistemi

Newton yönteminde her iterasyonda şu lineer sistem çözülür:

$$
H(z_k)\,\Delta z = \nabla \mathcal{L}(z_k)
$$

Burada:

$$
z = (x,y,\lambda)
$$

Güncelleme kuralı:

$$
z_{k+1} = z_k - \Delta z
$$

Yakınsama kriteri:

$$
\|\Delta z\| < \varepsilon
$$

---

## Yöntemin Özellikleri

- Eşitlik kısıtlı problemleri çözer
- Çözüm yakınında karesel yakınsama sağlar
- Hessian matrisinin terslenebilir olması gerekir
- Başlangıç noktasına duyarlıdır


In [7]:
import sympy as sp
import numpy as np
import plotly.graph_objects as go

In [8]:
def lagrange_newton_solver(f_func, g_func, baslangic_noktasi, tol=1e-6, max_iter=20):
    x_s, y_s, lam_s = sp.symbols('x y lambda')
  
    L = f_func(x_s, y_s) + lam_s * g_func(x_s, y_s)
    
    vars_s = [x_s, y_s, lam_s]
    grad_L = [L.diff(v) for v in vars_s]
    hess_L = [[grad.diff(v) for v in vars_s] for grad in grad_L]
    
    grad_numeric = sp.lambdify(vars_s, grad_L, 'numpy')
    hess_numeric = sp.lambdify(vars_s, hess_L, 'numpy')
     
    nokta = np.array(baslangic_noktasi, dtype=float) 
    
    for i in range(max_iter):
        g_val = np.array(grad_numeric(*nokta))
        h_val = np.array(hess_numeric(*nokta))
        
        step = np.linalg.solve(h_val, g_val) 
        nokta = nokta - step
        
        if np.linalg.norm(step) < tol:
            print(f"Yakınsama sağlandı! {i+1} adımda sonuç bulundu.")
            break
            
    return nokta 

In [9]:
def f_simple(x, y): return x**2 + y**2
def g_simple(x, y): return x + 2*y - 10

sonuc = lagrange_newton_solver(f_simple, g_simple, [1, 1, 0])

print("-" * 30)
print("BASİT LAGRANGE ÇÖZÜMÜ")
print("-" * 30)
print(f"Minimum Nokta: x = {sonuc[0]:.4f}, y = {sonuc[1]:.4f}")
print(f"Lagrange Çarpanı (lambda): {sonuc[2]:.4f}")
print(f"Kısıt Kontrolü (x+2y): {sonuc[0] + 2*sonuc[1]:.4f} (Hedef: 10)")

Yakınsama sağlandı! 2 adımda sonuç bulundu.
------------------------------
BASİT LAGRANGE ÇÖZÜMÜ
------------------------------
Minimum Nokta: x = 2.0000, y = 4.0000
Lagrange Çarpanı (lambda): -4.0000
Kısıt Kontrolü (x+2y): 10.0000 (Hedef: 10)


In [10]:
def visualize_constrained(f_func, g_func, sonuc_noktasi):
    
    x = np.linspace(-2, 4, 100)
    y = np.linspace(-2, 4, 100)
    X, Y = np.meshgrid(x, y)
    Z = f_func(X, Y)
    
    fig = go.Figure()
    
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.6, colorscale='Viridis', name='f(x,y)'))
     
    x_line = np.linspace(-1, 3, 100)
    y_line = 2 - x_line
    z_line = f_func(x_line, y_line)
    
    fig.add_trace(go.Scatter3d(x=x_line, y=y_line, z=z_line, 
                               mode='lines', line=dict(color='red', width=10),
                               name='Kısıt Doğrusu (x+y=2)'))
     
    fig.add_trace(go.Scatter3d(x=[sonuc_noktasi[0]], y=[sonuc_noktasi[1]], z=[f_func(sonuc_noktasi[0], sonuc_noktasi[1])],
                               mode='markers', marker=dict(size=10, color='orange'),
                               name='Kısıtlı Minimum'))

    fig.update_layout(title="Lagrange Kısıtlı Optimizasyon Görseli", scene=dict(aspectmode='cube'))
    fig.show()

In [11]:
def f_comp(x, y): return (x-2)**4 + (y-3)**4
def g_comp(x, y): return x**2 + y**2 - 1

sonuc = lagrange_newton_solver(f_comp, g_comp, [0.5, 0.5, 0])
 
x_grid = np.linspace(-1.5, 3.5, 100)
y_grid = np.linspace(-1.5, 4.5, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f_comp(X, Y)

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.5, colorscale='Electric', name='Hedef Fonksiyon'))
 
theta = np.linspace(0, 2*np.pi, 100)
xc, yc = np.cos(theta), np.sin(theta)
zc = f_comp(xc, yc)
fig.add_trace(go.Scatter3d(x=xc, y=yc, z=zc, mode='lines', line=dict(color='red', width=10), name='Birim Çember Kısıtı'))
 
 
fig.add_trace(go.Scatter3d(x=[sonuc[0]], y=[sonuc[1]], z=[f_comp(sonuc[0], sonuc[1])],
                           mode='markers', marker=dict(size=12, color='orange', symbol='diamond'), name='Kısıtlı Minimum'))

fig.update_layout(title="Karmaşık Lagrange: 4. Derece Fonksiyon ve Çember Kısıtı",
                  scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='f(x,y)'))
fig.show()

print(f"Karmaşık Çözüm: x={sonuc[0]:.4f}, y={sonuc[1]:.4f}")

Yakınsama sağlandı! 6 adımda sonuç bulundu.


Karmaşık Çözüm: x=0.4068, y=0.9135
